In [ ]:
import os
import h5py
from pathlib import Path
import jax
from jax import numpy as jnp
import jax.scipy.signal
from jax import config
from tqdm.auto import tqdm
import numpy as np  

config.update("jax_enable_x64", True)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true'

DATA_DIR = Path.cwd().parent.parent / "data" / "helmholtz_data"
FILE_PATH = DATA_DIR / "Helmholtz.h5"

In [ ]:
config.update("jax_enable_x64", True)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true'

DATA_DIR = Path.cwd().parent.parent / "data" / "helmholtz_data"
FILE_PATH = DATA_DIR / "Helmholtz.h5"

@jax.jit  
def generate_table(u_grid, dx=1.0, dy=1.0):
    """
    Takes any 2D simulation grid and returns a flattened (N, 5) array.
    dx: Physical distance between adjacent pixels on the x-axis.
    dy: Physical distance between adjacent pixels on the y-axis.
    """
    k_dx = jnp.array([[ 0.0, -1.0,  0.0],
                      [ 0.0,  0.0,  0.0],
                      [ 0.0,  1.0,  0.0]]) / (2.0 * dx)
                      
    k_dy = jnp.array([[ 0.0,  0.0,  0.0],
                      [-1.0,  0.0,  1.0],
                      [ 0.0,  0.0,  0.0]]) / (2.0 * dy)
                      
    k_dxx = jnp.array([[ 0.0,  1.0,  0.0],
                       [ 0.0, -2.0,  0.0],
                       [ 0.0,  1.0,  0.0]]) / (dx ** 2)
                       
    k_dyy = jnp.array([[ 0.0,  0.0,  0.0],
                       [ 1.0, -2.0,  1.0],
                       [ 0.0,  0.0,  0.0]]) / (dy ** 2)

    du_dx = jax.scipy.signal.correlate2d(u_grid, k_dx, mode='valid')
    du_dy = jax.scipy.signal.correlate2d(u_grid, k_dy, mode='valid')
    d2u_dx2 = jax.scipy.signal.correlate2d(u_grid, k_dxx, mode='valid')
    d2u_dy2 = jax.scipy.signal.correlate2d(u_grid, k_dyy, mode='valid')
    
    # strips 1 pixel from all edges to avoid sampling boundary pixels
    u_interior = u_grid[1:-1, 1:-1]
    
    # flatten and stack into columns: [u, du_dx, du_dy, d2u_dx2, d2u_dy2]
    final_table = jnp.column_stack((
        u_interior.flatten(), 
        du_dx.flatten(), 
        du_dy.flatten(), 
        d2u_dx2.flatten(), 
        d2u_dy2.flatten()
    ))
    
    return final_table

In [ ]:
# SANITY CHECK EXECUTION
# 128x128 grid from 0.0 to 1.0
nx, ny = 128, 128
x_coords = np.linspace(0.0, 1.0, nx, dtype=np.float64)
y_coords = np.linspace(0.0, 1.0, ny, dtype=np.float64)
X, Y = np.meshgrid(x_coords, y_coords, indexing='ij')

# Calculate the exact physical step size
dx = x_coords[1] - x_coords[0]
dy = y_coords[1] - y_coords[0]

# Generate a fake u(x,y) = x^3 + 2y^2
U_grid = X**3 + 2 * (Y**2)

# Pass it through dataset generator
discrete_table = generate_table(U_grid, dx=dx, dy=dy)

# Calculate the exact calculus derivatives for the interior pixels
X_int = X[1:-1, 1:-1].flatten()
Y_int = Y[1:-1, 1:-1].flatten()

true_u = X_int**3 + 2 * (Y_int**2)
true_dx = 3 * (X_int**2)
true_dy = 4 * Y_int
true_dxx = 6 * X_int
true_dyy = 4 * jnp.ones_like(Y_int)

# Compare the discrete approximation to the exact calculus
print("-" * 50)
print("MEAN ABSOLUTE ERROR (Numerical Stencil vs Pure Calculus)")
print("-" * 50)
print(f"u        : {jnp.mean(jnp.abs(discrete_table[:, 0] - true_u)):.2e}")
print(f"du/dx    : {jnp.mean(jnp.abs(discrete_table[:, 1] - true_dx)):.2e}")
print(f"du/dy    : {jnp.mean(jnp.abs(discrete_table[:, 2] - true_dy)):.2e}")
print(f"d2u/dx2  : {jnp.mean(jnp.abs(discrete_table[:, 3] - true_dxx)):.2e}")
print(f"d2u/dy2  : {jnp.mean(jnp.abs(discrete_table[:, 4] - true_dyy)):.2e}")

"""
Expected
u        : 0.00e+00
du/dx    : 6.20e-05
du/dy    : 0.00e+00
d2u/dx2  : 0.00e+00
d2u/dy2  : 0.00e+00
"""

In [ ]:
if __name__ == "__main__":    
    all_sample_tables = []
    
    with h5py.File(FILE_PATH, 'r') as f:
        sample_keys = list(f.keys())
        
        for key in tqdm(sample_keys):
            u_sim = jnp.array(f[key]['u'][:])
            
            # For a 128x128 grid. Adjust if needed.
            nx = 128
            dx = 1.0 / nx 
            
            sample_matrix = generate_table(u_sim, dx=dx, dy=dx)
            
            # Move data from GPU to CPU to prevent OOM on big datasets
            all_sample_tables.append(np.array(sample_matrix))
            
    print("Stacking all samples into final table...")
    full_dataset_matrix = np.vstack(all_sample_tables)
    
    print(f"Total Samples Processed: {len(sample_keys)}")
    print(f"Final Massive Table Shape: {full_dataset_matrix.shape}")
    print("\nFirst 5 rows of data (Columns: [u, du_dx, du_dy, d2u_dx2, d2u_dy2]):")
    
    with np.printoptions(precision=4, suppress=True):
        print(full_dataset_matrix[:5, :])